In [ ]:
!pip install torchvision
!pip install fvcore umap-learn scikit-video opencv-python-headless matplotlib seaborn tqdm scikit-learn imblearn albumentations

In [ ]:
# Install required dependencies
!pip install fvcore umap-learn scipy scikit-learn imbalanced-learn pytorchvideo

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.hub
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, confusion_matrix, roc_auc_score, silhouette_score, classification_report
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
import umap
from scipy.stats import kendalltau, spearmanr, entropy, wasserstein_distance
from scipy.spatial.distance import jensenshannon
from scipy.linalg import sqrtm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import warnings
from albumentations.pytorch import ToTensorV2
import albumentations as A
from collections import Counter

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Enhanced Configuration
CONFIG = {
    'data_folder': "/kaggle/input/summe-dataset/main_videos",
    'output_folder': "/kaggle/working",
    'seed': 42,
    'test_size': 0.2,
    'cv_folds': 5,
    'batch_size': 2,  # Reduced for stability
    'num_workers': 2,
    'frame_samples': 64,  # Increased for better temporal representation
    'img_size': 224,
    'use_data_augmentation': True,
    'use_ensemble': True,
    'use_smote': True,
    'feature_selection': True,
}

# Set random seeds for reproducibility
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(CONFIG['seed'])

# Function to get all video files with better error handling
def get_video_files(data_dir):
    video_files = []
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.wmv']
    
    if not os.path.exists(data_dir):
        logging.error(f"Data directory does not exist: {data_dir}")
        return video_files
    
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            files = []
            for f in os.listdir(class_path):
                if any(f.lower().endswith(ext) for ext in video_extensions):
                    full_path = os.path.join(class_path, f)
                    # Verify video can be opened
                    cap = cv2.VideoCapture(full_path)
                    if cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0:
                        files.append(full_path)
                    cap.release()
            
            video_files.extend([(f, class_name) for f in files])
            logging.info(f"Class '{class_name}': {len(files)} valid videos")
    
    logging.info(f"Total valid videos found: {len(video_files)}")
    return video_files

# Enhanced VideoDataset with better augmentation and feature extraction
class SupervisedVideoDataset(Dataset):
    def __init__(self, video_files, label_encoder, mode='train', transform=None):
        self.video_files = video_files
        self.label_encoder = label_encoder
        self.mode = mode
        self.transform = transform
        self.labels = [self.label_encoder.transform([class_name])[0] for _, class_name in self.video_files]
        
        # Enhanced augmentation pipeline
        if CONFIG['use_data_augmentation'] and mode == 'train':
            self.video_augmentation = A.Compose([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.5),
                A.GaussianBlur(blur_limit=3, p=0.3),
                A.Rotate(limit=15, p=0.3),
                A.HorizontalFlip(p=0.5),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        else:
            self.video_augmentation = A.Compose([
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        
        # Precompute all features
        self.temporal_features = self._compute_temporal_features()
        self.motion_features = self._compute_motion_features()
        self.texture_features = self._compute_texture_features()

    def _compute_temporal_features(self):
        features = []
        for video_path, _ in tqdm(self.video_files, desc=f"Computing temporal features ({self.mode})"):
            feature = self._extract_enhanced_temporal_features(video_path)
            features.append(feature)
        return np.array(features)
    
    def _extract_enhanced_temporal_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        n_samples = min(CONFIG['frame_samples'], total_frames)
        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        frames = []
        gray_frames = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame_resized = cv2.resize(frame, (CONFIG['img_size'], CONFIG['img_size']))
                frames.append(frame_resized)
                gray_frames.append(cv2.cvtColor(frame_resized, cv2.COLOR_BGR2GRAY))
        
        cap.release()
        
        if len(frames) < 2:
            return np.zeros(25)  # Increased feature size
        
        features = []
        
        # Enhanced optical flow analysis
        flows = []
        flow_magnitudes = []
        flow_angles = []
        
        for i in range(len(gray_frames) - 1):
            flow = cv2.calcOpticalFlowFarneback(
                gray_frames[i], gray_frames[i + 1], None, 
                0.5, 3, 15, 3, 5, 1.2, 0
            )
            mag, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            flows.append(flow)
            flow_magnitudes.append(mag)
            flow_angles.append(angle)
        
        if flows:
            # Flow magnitude statistics
            flow_stats = [
                np.mean([np.mean(f) for f in flow_magnitudes]),
                np.mean([np.std(f) for f in flow_magnitudes]),
                np.mean([np.max(f) for f in flow_magnitudes]),
                np.std([np.mean(f) for f in flow_magnitudes]),
                np.mean([np.percentile(f, 95) for f in flow_magnitudes]),  # 95th percentile
                np.mean([np.var(f) for f in flow_magnitudes]),  # Variance
            ]
            
            # Flow direction statistics
            angle_stats = [
                np.mean([np.mean(a) for a in flow_angles]),
                np.std([np.mean(a) for a in flow_angles]),
                np.mean([np.std(a) for a in flow_angles]),
            ]
            
            features.extend(flow_stats + angle_stats)
        else:
            features.extend([0] * 9)
        
        # Enhanced frame difference analysis
        frame_diffs = []
        for i in range(len(frames) - 1):
            diff = cv2.absdiff(frames[i], frames[i + 1])
            frame_diffs.append([
                np.mean(diff),
                np.std(diff),
                np.max(diff),
                np.percentile(diff, 95)
            ])
        
        if frame_diffs:
            diff_array = np.array(frame_diffs)
            diff_stats = [
                np.mean(diff_array[:, 0]),  # Mean of means
                np.std(diff_array[:, 0]),   # Std of means
                np.mean(diff_array[:, 1]),  # Mean of stds
                np.mean(diff_array[:, 2]),  # Mean of maxes
                np.mean(diff_array[:, 3]),  # Mean of 95th percentiles
                np.std(diff_array[:, 0]),   # Temporal consistency
            ]
        else:
            diff_stats = [0] * 6
        
        features.extend(diff_stats)
        
        # Video metadata features
        metadata_features = [
            total_frames,
            fps if fps > 0 else 25,  # Default FPS if invalid
            total_frames / max(fps, 1),  # Duration
            width * height,  # Resolution
            width / max(height, 1),  # Aspect ratio
        ]
        
        features.extend(metadata_features)
        
        # Temporal smoothness
        if len(flow_magnitudes) > 1:
            temporal_smoothness = np.corrcoef([np.mean(f) for f in flow_magnitudes[:-1]], 
                                            [np.mean(f) for f in flow_magnitudes[1:]])[0, 1]
            if np.isnan(temporal_smoothness):
                temporal_smoothness = 0
        else:
            temporal_smoothness = 0
        
        features.append(temporal_smoothness)
        
        # Scene change detection
        scene_changes = 0
        if len(frame_diffs) > 0:
            diff_means = [np.mean(diff) for diff in frame_diffs]
            threshold = np.mean(diff_means) + 2 * np.std(diff_means)
            scene_changes = np.sum(np.array([np.mean(diff) for diff in frame_diffs]) > threshold)
        
        features.append(scene_changes / max(len(frame_diffs), 1))
        
        # Edge density analysis
        edge_densities = []
        for frame in frames[::4]:  # Sample every 4th frame
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            edge_density = np.sum(edges > 0) / (edges.shape[0] * edges.shape[1])
            edge_densities.append(edge_density)
        
        if edge_densities:
            features.extend([np.mean(edge_densities), np.std(edge_densities)])
        else:
            features.extend([0, 0])
        
        return np.array(features, dtype=np.float32)

    def _compute_motion_features(self):
        features = []
        for video_path, _ in tqdm(self.video_files, desc=f"Computing motion features ({self.mode})"):
            feature = self._extract_motion_features(video_path)
            features.append(feature)
        return np.array(features)
    
    def _extract_motion_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Sample frames for motion analysis
        n_samples = min(32, total_frames)
        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        prev_frame = None
        motion_vectors = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                gray = cv2.resize(gray, (224, 224))
                
                if prev_frame is not None:
                    flow = cv2.calcOpticalFlowFarneback(prev_frame, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                    magnitude = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
                    motion_vectors.append(magnitude)
                
                prev_frame = gray
        
        cap.release()
        
        if not motion_vectors:
            return np.zeros(8)
        
        # Motion statistics
        all_magnitudes = np.concatenate(motion_vectors)
        features = [
            np.mean(all_magnitudes),
            np.std(all_magnitudes),
            np.max(all_magnitudes),
            np.min(all_magnitudes),
            np.percentile(all_magnitudes, 75),
            np.percentile(all_magnitudes, 25),
            np.sum(all_magnitudes > np.mean(all_magnitudes)) / len(all_magnitudes),  # Activity ratio
            len(motion_vectors)  # Number of motion frames
        ]
        
        return np.array(features, dtype=np.float32)

    def _compute_texture_features(self):
        features = []
        for video_path, _ in tqdm(self.video_files, desc=f"Computing texture features ({self.mode})"):
            feature = self._extract_texture_features(video_path)
            features.append(feature)
        return np.array(features)
    
    def _extract_texture_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Sample fewer frames for texture analysis
        n_samples = min(8, total_frames)
        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        texture_features = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                gray = cv2.resize(gray, (224, 224))
                
                # Local Binary Pattern
                from skimage.feature import local_binary_pattern
                lbp = local_binary_pattern(gray, 8, 1, method='uniform')
                lbp_hist, _ = np.histogram(lbp.ravel(), bins=10)
                lbp_hist = lbp_hist.astype(float)
                lbp_hist /= (lbp_hist.sum() + 1e-8)
                
                texture_features.append(lbp_hist)
        
        cap.release()
        
        if not texture_features:
            return np.zeros(10)
        
        # Average texture features across frames
        avg_texture = np.mean(texture_features, axis=0)
        return avg_texture.astype(np.float32)

    def __len__(self):
        return len(self.video_files)
    
    def __getitem__(self, idx):
        video_path, class_name = self.video_files[idx]
        
        try:
            frames = self._extract_frames_robust(video_path)
            label = self.label_encoder.transform([class_name])[0]
            temporal_feat = self.temporal_features[idx]
            motion_feat = self.motion_features[idx]
            texture_feat = self.texture_features[idx]
            
            # Combine all features
            combined_features = np.concatenate([temporal_feat, motion_feat, texture_feat])
            
            return frames, label, combined_features, video_path
        except Exception as e:
            logging.warning(f"Error processing video {video_path}: {e}")
            dummy_frames = torch.zeros(3, CONFIG['frame_samples'], CONFIG['img_size'], CONFIG['img_size'])
            dummy_label = 0
            dummy_features = np.zeros(43)  # Updated feature size
            return dummy_frames, dummy_label, dummy_features, video_path
    
    def _extract_frames_robust(self, video_path, max_retries=3):
        for attempt in range(max_retries):
            try:
                cap = cv2.VideoCapture(video_path)
                if not cap.isOpened():
                    raise ValueError(f"Could not open video: {video_path}")
                
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if total_frames == 0:
                    raise ValueError(f"Video has no frames: {video_path}")
                
                # Sample frames more robustly
                n_samples = CONFIG['frame_samples']
                if total_frames < n_samples:
                    indices = list(range(total_frames))
                    # Pad with repeated frames
                    while len(indices) < n_samples:
                        indices.extend(indices[:n_samples-len(indices)])
                else:
                    indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
                
                frames = []
                
                for idx in indices:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                    ret, frame = cap.read()
                    if ret and frame is not None:
                        frame = cv2.resize(frame, (CONFIG['img_size'], CONFIG['img_size']))
                        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        
                        # Apply augmentation
                        try:
                            augmented = self.video_augmentation(image=frame)
                            frame_tensor = augmented['image']
                        except:
                            # Fallback to basic normalization
                            frame = frame.astype(np.float32) / 255.0
                            frame = (frame - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
                            frame_tensor = torch.from_numpy(frame.transpose(2, 0, 1))
                        
                        frames.append(frame_tensor)
                
                cap.release()
                
                # Ensure we have the right number of frames
                while len(frames) < n_samples:
                    if frames:
                        frames.append(frames[-1].clone())
                    else:
                        frames.append(torch.zeros(3, CONFIG['img_size'], CONFIG['img_size']))
                
                frames = frames[:n_samples]
                return torch.stack(frames).permute(1, 0, 2, 3)  # [C, T, H, W]
                
            except Exception as e:
                logging.warning(f"Attempt {attempt + 1} failed for {video_path}: {e}")
                if attempt == max_retries - 1:
                    raise e

# VideoFeatureExtractor using SlowFast R50
class VideoFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super(VideoFeatureExtractor, self).__init__()
        # Load SlowFast R50 model from PyTorchVideo via torch.hub
        self.model = torch.hub.load('facebookresearch/pytorchvideo:main', 'slowfast_r50', pretrained=pretrained)
        # Remove the classification head to get features
        self.model.blocks[-1] = nn.Identity()

    def forward(self, x):
        slow_frames = x[:, :, ::4, :, :]  # Take every 4th frame (~8 frames)
        fast_frames = x  # Use all 32 frames for fast pathway
        inputs = [slow_frames, fast_frames]
        features = self.model(inputs)
        if features.dim() > 2:
            features = features.view(features.size(0), -1)
        return features

# Enhanced Feature Extraction with better error handling
def extract_features(model, loader, device='cuda'):
    model.to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model.eval()
    visual_features, combined_features, labels, video_paths = [], [], [], []
    
    with torch.no_grad():
        for batch_idx, (frames, lbls, combined_feats, paths) in enumerate(tqdm(loader, desc="Extracting Features")):
            try:
                frames = frames.to(device)
                vis_feats = model(frames)
                
                # Normalize visual features
                vis_feats = F.normalize(vis_feats, p=2, dim=1)
                
                visual_features.append(vis_feats.cpu().numpy())
                combined_features.append(combined_feats.numpy())
                labels.extend(lbls.numpy())
                video_paths.extend(paths)
                
            except Exception as e:
                logging.warning(f"Error processing batch {batch_idx}: {e}")
                # Add dummy features for failed batch
                batch_size = len(lbls)
                dummy_vis = np.zeros((batch_size, vis_feats.shape[1] if 'vis_feats' in locals() else 1024))
                visual_features.append(dummy_vis)
                combined_features.append(combined_feats.numpy())
                labels.extend(lbls.numpy())
                video_paths.extend(paths)
    
    visual_features = np.vstack(visual_features)
    combined_features = np.vstack(combined_features)
    
    # Combine visual and traditional features
    all_features = np.hstack([visual_features, combined_features])
    
    return all_features, combined_features, np.array(labels), video_paths

# Enhanced Ensemble Classifier
class EnhancedEnsembleClassifier:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.scalers = {}
        self.classifiers = {}
        self.feature_selector = None
        
    def _create_classifiers(self, n_classes):
        """Create diverse set of classifiers"""
        classifiers = {
            'rf': RandomForestClassifier(
                n_estimators=200, 
                max_depth=10, 
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=self.random_state,
                class_weight='balanced'
            ),
            'gb': GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                random_state=self.random_state
            ),
            'svm': SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                probability=True,
                random_state=self.random_state,
                class_weight='balanced'
            ),
            'knn': KNeighborsClassifier(
                n_neighbors=min(5, max(3, n_classes * 2)),
                weights='distance',
                metric='minkowski'
            ),
            'lr': LogisticRegression(
                random_state=self.random_state,
                max_iter=1000,
                class_weight='balanced'
            )
        }
        return classifiers
    
    def fit(self, X_train, y_train, use_smote=True, feature_selection=True):
        """Fit ensemble with preprocessing"""
        
        # Feature selection
        if feature_selection and CONFIG['feature_selection']:
            from sklearn.feature_selection import SelectKBest, f_classif
            k = min(20, X_train.shape[1] // 2)  # Select top features
            self.feature_selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = self.feature_selector.fit_transform(X_train, y_train)
        else:
            X_train_selected = X_train
        
        # Handle class imbalance
        if use_smote and CONFIG['use_smote']:
            try:
                # Try SMOTE first
                smote = SMOTE(random_state=self.random_state, k_neighbors=min(3, len(np.unique(y_train))-1))
                X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
                logging.info(f"Applied SMOTE: {Counter(y_train)} -> {Counter(y_train_balanced)}")
            except:
                try:
                    # Fallback to ADASYN
                    adasyn = ADASYN(random_state=self.random_state, n_neighbors=min(3, len(np.unique(y_train))-1))
                    X_train_balanced, y_train_balanced = adasyn.fit_resample(X_train_selected, y_train)
                    logging.info(f"Applied ADASYN: {Counter(y_train)} -> {Counter(y_train_balanced)}")
                except:
                    # No resampling if both fail
                    X_train_balanced, y_train_balanced = X_train_selected, y_train
                    logging.warning("SMOTE/ADASYN failed, using original data")
        else:
            X_train_balanced, y_train_balanced = X_train_selected, y_train
        
        # Create and train classifiers
        n_classes = len(np.unique(y_train))
        base_classifiers = self._create_classifiers(n_classes)
        
        for name, clf in base_classifiers.items():
            try:
                # Scale features for each classifier
                if name in ['svm', 'knn', 'lr']:
                    scaler = RobustScaler()
                    X_scaled = scaler.fit_transform(X_train_balanced)
                    self.scalers[name] = scaler
                else:
                    X_scaled = X_train_balanced
                    self.scalers[name] = None
                
                # Train classifier
                clf.fit(X_scaled, y_train_balanced)
                self.classifiers[name] = clf
                
                logging.info(f"Trained {name} classifier successfully")
                
            except Exception as e:
                logging.warning(f"Failed to train {name}: {e}")
        
        # Create voting classifier with successful models
        if len(self.classifiers) > 1 and CONFIG['use_ensemble']:
            voting_estimators = []
            for name, clf in self.classifiers.items():
                voting_estimators.append((name, clf))
            
            self.voting_classifier = VotingClassifier(
                estimators=voting_estimators,
                voting='soft'
            )
            
            # For voting classifier, use the same preprocessing as the best individual classifier
            best_scaler = self.scalers.get('rf', None)  # Use RF scaler as default
            if best_scaler:
                X_voting = best_scaler.transform(X_train_balanced)
            else:
                X_voting = X_train_balanced
            
            try:
                self.voting_classifier.fit(X_voting, y_train_balanced)
                self.best_scaler = best_scaler
                logging.info("Trained ensemble voting classifier")
            except:
                self.voting_classifier = None
                logging.warning("Failed to create voting classifier")
        else:
            self.voting_classifier = None
    
    def predict(self, X_test):
        """Predict using ensemble with proper preprocessing"""
        
        # Apply feature selection if used during training
        if self.feature_selector:
            X_test_selected = self.feature_selector.transform(X_test)
        else:
            X_test_selected = X_test
        
        predictions = {}
        probabilities = {}
        
        # Get predictions from individual classifiers
        for name, clf in self.classifiers.items():
            try:
                # Apply same scaling as used during training
                if self.scalers[name]:
                    X_scaled = self.scalers[name].transform(X_test_selected)
                else:
                    X_scaled = X_test_selected
                
                pred = clf.predict(X_scaled)
                prob = clf.predict_proba(X_scaled)
                
                predictions[name] = pred
                probabilities[name] = prob
                
            except Exception as e:
                logging.warning(f"Failed to predict with {name}: {e}")
        
        # Use voting classifier if available
        if self.voting_classifier:
            try:
                if self.best_scaler:
                    X_voting = self.best_scaler.transform(X_test_selected)
                else:
                    X_voting = X_test_selected
                
                ensemble_pred = self.voting_classifier.predict(X_voting)
                ensemble_prob = self.voting_classifier.predict_proba(X_voting)
                
                return ensemble_pred, ensemble_prob, predictions, probabilities
            except:
                logging.warning("Voting classifier failed, using best individual classifier")
        
        # Fallback to best individual classifier
        if 'rf' in predictions:
            return predictions['rf'], probabilities['rf'], predictions, probabilities
        elif predictions:
            best_name = list(predictions.keys())[0]
            return predictions[best_name], probabilities[best_name], predictions, probabilities
        else:
            # Ultimate fallback
            dummy_pred = np.zeros(X_test.shape[0], dtype=int)
            dummy_prob = np.zeros((X_test.shape[0], 2))
            dummy_prob[:, 0] = 1  # All predict class 0
            return dummy_pred, dummy_prob, {}, {}

    def get_feature_importance(self):
        """Get feature importance from tree-based models"""
        importance_dict = {}
        
        for name, clf in self.classifiers.items():
            if hasattr(clf, 'feature_importances_'):
                importance_dict[name] = clf.feature_importances_
        
        return importance_dict

# Advanced Evaluation Metrics
class AdvancedMetrics:
    @staticmethod
    def compute_all_metrics(y_true, y_pred, y_prob=None, labels=None):
        """Compute comprehensive evaluation metrics"""
        metrics = {}
        
        # Basic metrics
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
        metrics['f1_macro'] = f1_score(y_true, y_pred, average='macro')
        metrics['f1_weighted'] = f1_score(y_true, y_pred, average='weighted')
        metrics['precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
        
        # AUC metrics (if probabilities available)
        if y_prob is not None and len(np.unique(y_true)) == 2:
            try:
                metrics['roc_auc'] = roc_auc_score(y_true, y_prob[:, 1])
            except:
                metrics['roc_auc'] = 0.5
        elif y_prob is not None:
            try:
                metrics['roc_auc_ovr'] = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
            except:
                metrics['roc_auc_ovr'] = 0.5
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        metrics['confusion_matrix'] = cm
        
        # Per-class metrics
        report = classification_report(y_true, y_pred, target_names=labels, output_dict=True, zero_division=0)
        metrics['classification_report'] = report
        
        return metrics
    
    @staticmethod
    def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
        """Plot confusion matrix"""
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=labels, yticklabels=labels)
        plt.title(title)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_feature_importance(importance_dict, feature_names=None, top_k=20):
        """Plot feature importance from multiple models"""
        if not importance_dict:
            return
        
        fig, axes = plt.subplots(len(importance_dict), 1, figsize=(12, 4*len(importance_dict)))
        if len(importance_dict) == 1:
            axes = [axes]
        
        for idx, (model_name, importance) in enumerate(importance_dict.items()):
            if feature_names is None:
                feature_names = [f'Feature_{i}' for i in range(len(importance))]
            
            # Get top k features
            top_indices = np.argsort(importance)[-top_k:]
            top_importance = importance[top_indices]
            top_names = [feature_names[i] for i in top_indices]
            
            axes[idx].barh(range(len(top_importance)), top_importance)
            axes[idx].set_yticks(range(len(top_importance)))
            axes[idx].set_yticklabels(top_names)
            axes[idx].set_title(f'{model_name} - Top {top_k} Feature Importance')
            axes[idx].set_xlabel('Importance')
        
        plt.tight_layout()
        plt.show()

# Cross-validation with proper error handling
def perform_cross_validation(X, y, model, cv_folds=5, random_state=42):
    """Perform stratified cross-validation"""
    try:
        skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
        
        cv_scores = {}
        cv_scores['accuracy'] = []
        cv_scores['f1_macro'] = []
        cv_scores['balanced_accuracy'] = []
        
        fold_predictions = []
        fold_true_labels = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            try:
                X_train_fold, X_val_fold = X[train_idx], X[val_idx]
                y_train_fold, y_val_fold = y[train_idx], y[val_idx]
                
                # Create new model instance for each fold
                fold_model = EnhancedEnsembleClassifier(random_state=random_state)
                fold_model.fit(X_train_fold, y_train_fold)
                
                # Predict
                y_pred, y_prob, _, _ = fold_model.predict(X_val_fold)
                
                # Calculate metrics
                acc = accuracy_score(y_val_fold, y_pred)
                f1 = f1_score(y_val_fold, y_pred, average='macro')
                bal_acc = balanced_accuracy_score(y_val_fold, y_pred)
                
                cv_scores['accuracy'].append(acc)
                cv_scores['f1_macro'].append(f1)
                cv_scores['balanced_accuracy'].append(bal_acc)
                
                fold_predictions.extend(y_pred)
                fold_true_labels.extend(y_val_fold)
                
                logging.info(f"Fold {fold+1}: Acc={acc:.3f}, F1={f1:.3f}, Bal_Acc={bal_acc:.3f}")
                
            except Exception as e:
                logging.warning(f"Error in fold {fold+1}: {e}")
                # Add dummy scores to maintain fold count
                cv_scores['accuracy'].append(0.0)
                cv_scores['f1_macro'].append(0.0)
                cv_scores['balanced_accuracy'].append(0.0)
        
        # Calculate mean and std
        cv_results = {}
        for metric, scores in cv_scores.items():
            cv_results[f'{metric}_mean'] = np.mean(scores)
            cv_results[f'{metric}_std'] = np.std(scores)
        
        return cv_results, fold_predictions, fold_true_labels
        
    except Exception as e:
        logging.error(f"Cross-validation failed: {e}")
        return {}, [], []

# Dimensionality Reduction and Visualization
def perform_dimensionality_reduction(X, y, labels, methods=['pca', 'tsne', 'umap']):
    """Perform multiple dimensionality reduction techniques"""
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    results = {}
    
    if 'pca' in methods:
        try:
            pca = PCA(n_components=2, random_state=42)
            X_pca = pca.fit_transform(X_scaled)
            results['pca'] = {
                'embedding': X_pca,
                'explained_variance': pca.explained_variance_ratio_
            }
        except Exception as e:
            logging.warning(f"PCA failed: {e}")
    
    if 'tsne' in methods:
        try:
            tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X)//4))
            X_tsne = tsne.fit_transform(X_scaled)
            results['tsne'] = {'embedding': X_tsne}
        except Exception as e:
            logging.warning(f"t-SNE failed: {e}")
    
    if 'umap' in methods:
        try:
            umap_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=min(15, len(X)//3))
            X_umap = umap_reducer.fit_transform(X_scaled)
            results['umap'] = {'embedding': X_umap}
        except Exception as e:
            logging.warning(f"UMAP failed: {e}")
    
    # Plot results
    n_methods = len(results)
    if n_methods > 0:
        fig, axes = plt.subplots(1, n_methods, figsize=(6*n_methods, 5))
        if n_methods == 1:
            axes = [axes]
        
        colors = plt.cm.tab10(np.linspace(0, 1, len(np.unique(y))))
        
        for idx, (method, data) in enumerate(results.items()):
            embedding = data['embedding']
            
            for class_idx, label in enumerate(labels):
                mask = y == class_idx
                axes[idx].scatter(embedding[mask, 0], embedding[mask, 1], 
                                c=[colors[class_idx]], label=label, alpha=0.7)
            
            axes[idx].set_title(f'{method.upper()} Visualization')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)
            
            if method == 'pca':
                var_explained = data['explained_variance']
                axes[idx].set_xlabel(f'PC1 ({var_explained[0]:.1%} variance)')
                axes[idx].set_ylabel(f'PC2 ({var_explained[1]:.1%} variance)')
        
        plt.tight_layout()
        plt.show()
    
    return results

# Main execution pipeline
def main():
    """Main execution pipeline with comprehensive error handling"""
    
    print("="*60)
    print("Enhanced Video Classification System")
    print("="*60)
    
    # Check if data directory exists
    if not os.path.exists(CONFIG['data_folder']):
        print(f"Error: Data folder '{CONFIG['data_folder']}' not found!")
        print("Please update CONFIG['data_folder'] with the correct path.")
        return
    
    # Get video files
    print("\n1. Loading video files...")
    video_files = get_video_files(CONFIG['data_folder'])
    
    if len(video_files) == 0:
        print("No valid video files found!")
        return
    
    # Check class distribution
    classes = [class_name for _, class_name in video_files]
    class_counts = Counter(classes)
    print(f"\nClass distribution: {dict(class_counts)}")
    
    if len(class_counts) < 2:
        print("Error: Need at least 2 classes for classification!")
        return
    
    # Create label encoder
    label_encoder = LabelEncoder()
    label_encoder.fit(list(class_counts.keys()))
    class_labels = list(label_encoder.classes_)
    
    print(f"Classes: {class_labels}")
    
    # Split data
    print("\n2. Splitting data...")
    train_files, test_files = train_test_split(
        video_files, 
        test_size=CONFIG['test_size'], 
        random_state=CONFIG['seed'],
        stratify=[class_name for _, class_name in video_files]
    )
    
    print(f"Train: {len(train_files)} videos")
    print(f"Test: {len(test_files)} videos")
    
    # Create datasets
    print("\n3. Creating datasets...")
    train_dataset = SupervisedVideoDataset(train_files, label_encoder, mode='train')
    test_dataset = SupervisedVideoDataset(test_files, label_encoder, mode='test')
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    # Initialize feature extractor
    print("\n4. Initializing feature extractor...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    feature_extractor = VideoFeatureExtractor(pretrained=True)
    
    # Extract features
    print("\n5. Extracting features...")
    print("Extracting training features...")
    train_features, train_combined_features, train_labels, train_paths = extract_features(
        feature_extractor, train_loader, device
    )
    
    print("Extracting test features...")
    test_features, test_combined_features, test_labels, test_paths = extract_features(
        feature_extractor, test_loader, device
    )
    
    print(f"Training features shape: {train_features.shape}")
    print(f"Test features shape: {test_features.shape}")
    
    # Feature analysis
    print("\n6. Feature analysis...")
    print(f"Feature statistics:")
    print(f"  Mean: {np.mean(train_features):.4f}")
    print(f"  Std: {np.std(train_features):.4f}")
    print(f"  Min: {np.min(train_features):.4f}")
    print(f"  Max: {np.max(train_features):.4f}")
    
    # Check for NaN or infinite values
    nan_count = np.sum(np.isnan(train_features))
    inf_count = np.sum(np.isinf(train_features))
    if nan_count > 0 or inf_count > 0:
        print(f"Warning: Found {nan_count} NaN and {inf_count} infinite values")
        train_features = np.nan_to_num(train_features, nan=0.0, posinf=1.0, neginf=-1.0)
        test_features = np.nan_to_num(test_features, nan=0.0, posinf=1.0, neginf=-1.0)
    
    # Dimensionality reduction and visualization
    print("\n7. Dimensionality reduction and visualization...")
    dr_results = perform_dimensionality_reduction(
        train_features, train_labels, class_labels, methods=['pca', 'tsne', 'umap']
    )
    
    # Cross-validation
    print("\n8. Performing cross-validation...")
    cv_model = EnhancedEnsembleClassifier(random_state=CONFIG['seed'])
    cv_results, cv_predictions, cv_true_labels = perform_cross_validation(
        train_features, train_labels, cv_model, cv_folds=CONFIG['cv_folds']
    )
    
    if cv_results:
        print("Cross-validation results:")
        for metric, value in cv_results.items():
            print(f"  {metric}: {value:.4f}")
    
    # Train final model
    print("\n9. Training final model...")
    final_model = EnhancedEnsembleClassifier(random_state=CONFIG['seed'])
    final_model.fit(train_features, train_labels, use_smote=CONFIG['use_smote'])
    
    # Make predictions
    print("\n10. Making predictions...")
    test_predictions, test_probabilities, individual_predictions, individual_probabilities = final_model.predict(test_features)
    
    # Evaluate model
    print("\n11. Evaluating model...")
    metrics = AdvancedMetrics.compute_all_metrics(
        test_labels, test_predictions, test_probabilities, class_labels
    )
    
    print("\nTest Results:")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"F1 Score (Macro): {metrics['f1_macro']:.4f}")
    print(f"F1 Score (Weighted): {metrics['f1_weighted']:.4f}")
    print(f"Precision (Macro): {metrics['precision_macro']:.4f}")
    print(f"Recall (Macro): {metrics['recall_macro']:.4f}")
    print(f"Matthews Correlation Coefficient: {metrics['mcc']:.4f}")
    
    if 'roc_auc' in metrics:
        print(f"ROC AUC: {metrics['roc_auc']:.4f}")
    if 'roc_auc_ovr' in metrics:
        print(f"ROC AUC (OvR): {metrics['roc_auc_ovr']:.4f}")
    
    # Plot confusion matrix
    print("\n12. Plotting confusion matrix...")
    AdvancedMetrics.plot_confusion_matrix(
        metrics['confusion_matrix'], class_labels, "Test Set Confusion Matrix"
    )
    
    # Feature importance
    print("\n13. Analyzing feature importance...")
    importance_dict = final_model.get_feature_importance()
    if importance_dict:
        # Create feature names
        n_visual = train_features.shape[1] - train_combined_features.shape[1]
        feature_names = [f'Visual_{i}' for i in range(n_visual)]
        feature_names.extend([f'Temporal_{i}' for i in range(25)])  # Temporal features
        feature_names.extend([f'Motion_{i}' for i in range(8)])     # Motion features
        feature_names.extend([f'Texture_{i}' for i in range(10)])   # Texture features
        
        # Adjust if we used feature selection
        if final_model.feature_selector:
            selected_indices = final_model.feature_selector.get_support(indices=True)
            feature_names = [feature_names[i] for i in selected_indices]
        
        AdvancedMetrics.plot_feature_importance(importance_dict, feature_names, top_k=15)
    
    # Individual classifier performance
    print("\n14. Individual classifier performance:")
    for clf_name, pred in individual_predictions.items():
        try:
            acc = accuracy_score(test_labels, pred)
            f1 = f1_score(test_labels, pred, average='macro')
            print(f"  {clf_name}: Accuracy={acc:.4f}, F1={f1:.4f}")
        except:
            print(f"  {clf_name}: Failed to compute metrics")
    
    # Save results
    print("\n15. Saving results...")
    results_dict = {
        'config': CONFIG,
        'class_labels': class_labels,
        'test_metrics': metrics,
        'cv_results': cv_results,
        'test_predictions': test_predictions.tolist(),
        'test_true_labels': test_labels.tolist(),
        'test_video_paths': test_paths,
        'individual_predictions': {k: v.tolist() for k, v in individual_predictions.items()}
    }
    
    import json
    with open(os.path.join(CONFIG['output_folder'], 'classification_results.json'), 'w') as f:
        json.dump(results_dict, f, indent=2)
    
    print(f"Results saved to: {CONFIG['output_folder']}/classification_results.json")
    
    # Final summary
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    print(f"Dataset: {len(video_files)} videos, {len(class_labels)} classes")
    print(f"Best Test Accuracy: {metrics['accuracy']:.4f}")
    print(f"Best Test F1 (Macro): {metrics['f1_macro']:.4f}")
    print(f"Best Test Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    
    if cv_results:
        print(f"CV Accuracy: {cv_results.get('accuracy_mean', 0):.4f} ± {cv_results.get('accuracy_std', 0):.4f}")
        print(f"CV F1 (Macro): {cv_results.get('f1_macro_mean', 0):.4f} ± {cv_results.get('f1_macro_std', 0):.4f}")
    
    print("="*60)
    
    return final_model, metrics, results_dict

# Error handling wrapper
def safe_main():
    """Main function with comprehensive error handling"""
    try:
        return main()
    except KeyboardInterrupt:
        print("\nProcess interrupted by user")
        return None, None, None
    except Exception as e:
        logging.error(f"Fatal error in main execution: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

# Run the complete pipeline
if __name__ == "__main__":
    # Clear GPU cache if available
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"CUDA available: {torch.cuda.get_device_name()}")
        print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Execute main pipeline
    model, metrics, results = safe_main()
    
    if model is not None:
        print("\n✅ Pipeline completed successfully!")
    else:
        print("\n❌ Pipeline failed. Check logs for details.")